# Mini-project C  Routing citizen requests in Arabic and English

**Case:** a contact centre receives requests in Arabic and English and wants to send each one to the responsible unit automatically, and the unclear ones to a person.

**What your team presents (3 minutes, one point each):** what the model does, how well it works, where it fails, the value for service users, where a person reviews the output, and your recommendation. The last step of this notebook builds the summary for you.

To keep your choices, use *File, Save a copy in Drive* first.

**The data used here:**
- `citizen_requests.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/citizen_requests.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/citizen_requests.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

In [ ]:
#@title Step 1  The requests
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
df = pd.read_csv(DATA + "citizen_requests.csv"); CASE = "Routing citizen requests"
big((f"{len(df):,}", "requests"), ((df["language"] == "Arabic").sum(), "in Arabic"), ((df["language"] == "English").sum(), "in English"))
display(df.groupby("responsible_unit").size().to_frame("requests"))
display(df[["language", "text", "responsible_unit"]].sample(6, random_state=3))

In [ ]:
#@title Step 2  Build your router  (choose, then click &#9654;)
method = "Word pieces (works across spellings)" #@param ["Word pieces (works across spellings)", "Whole words"]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2) if method.startswith("Word pieces") else TfidfVectorizer(min_df=2)
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["responsible_unit"], test_size=0.2, random_state=42, stratify=df["responsible_unit"])
router = Pipeline([("text to numbers", vec), ("classifier", LogisticRegression(max_iter=3000))]).fit(X_train, y_train)
pred = router.predict(X_test); lang = df.loc[X_test.index, "language"]
big((f"{accuracy_score(y_test, pred):.1%}", "routed correctly"), (f"{accuracy_score(y_test[lang == 'Arabic'], pred[lang == 'Arabic']):.1%}", "Arabic"), (f"{accuracy_score(y_test[lang == 'English'], pred[lang == 'English']):.1%}", "English"))
box("Try changes", "Switch the method on the right and click &#9654; again. Which one works better for Arabic?", "try")

In [ ]:
#@title Step 3  Try your own request  (type a request on the right, then click &#9654;)
request = "متى سيتم استرداد رسوم التأشيرة؟" #@param {type:"string"}
pr = pd.Series(router.predict_proba([request])[0], index=router.classes_).sort_values(ascending=False)
bars(pr, "Which unit? (probability)", highlight=pr.index[0], fmt="{:.0%}")
box(f"Routed to: {pr.index[0]}", f"with {pr.iloc[0]:.0%} confidence. Type another request in Arabic or English on the right and click &#9654;.", "see")

In [ ]:
#@title Step 4  When should a person decide?  (then move the slider)
P = router.predict_proba(X_test); conf = P.max(axis=1); top = router.classes_[P.argmax(axis=1)]
def show(confidence):
    auto = conf >= confidence
    big((f"{auto.mean():.0%}", "routed automatically"), (f"{(top[auto] == y_test.values[auto]).mean() if auto.any() else 0:.1%}", "of those correct", GREEN), (f"{(~auto).mean():.0%}", "sent to a person"))
    box("Do you see it?", "Raise the bar and fewer requests are routed automatically, but those that are, are almost always right. The rest go to a person. Your team chooses the balance.", "see")
_ = live(show, confidence=fslider("Minimum confidence", 0.2, 0.95, 0.05, 0.5))

In [ ]:
#@title Step 5  Your team's summary  (fill in the fields on the right, then click &#9654;)
what_it_does = "" #@param {type:"string"}
how_well = "" #@param {type:"string"}
where_it_fails = "" #@param {type:"string"}
value_for_users = "" #@param {type:"string"}
where_a_person_reviews = "" #@param {type:"string"}
recommendation = "" #@param {type:"string"}
pts = [("What the model does", what_it_does), ("How well it works", how_well), ("Where it fails", where_it_fails),
       ("Value for service users", value_for_users), ("Where a person reviews the output", where_a_person_reviews), ("Our recommendation", recommendation)]
html = "".join(f'<div style="border-top:1px solid #d9d9d6;padding:10px 0"><div style="font-size:14px;color:#6e6e6a">{i + 1}  {t}</div>'
               f'<div style="font-size:19px">{v or "<i style=color:#A12527>to fill in</i>"}</div></div>' for i, (t, v) in enumerate(pts))
display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;max-width:900px;border:2px solid #1d1d1b;padding:18px 24px;color:#1d1d1b;background:#fff">'
             f'<div style="font-size:24px;font-weight:bold;margin-bottom:6px">Our team: {CASE}</div>{html}</div>'))